# Lab 04 — Follow a real request from bytes to completed response

First decode a deliberately fragmented UTF-8/SSE example on CPU. Then inspect a **recorded** local API request with literal prompts and actual tokenizer IDs. Finally execute one bounded vLLM API smoke yourself. This is not a judge, throughput test or concurrency-capacity measurement.

Read [Book 04](https://levshaazz.github.io/llm-serving-mastery/en/book/04/) and the [student guide](https://levshaazz.github.io/llm-serving-mastery/en/start/). Students arrange a CUDA GPU; use a private repository Issue with @levshaazz for diagnostic artifacts. Never expose this teaching server to a public interface.

**Unchanged scope:** pinned 0.5B local snapshot, 2048-token context, four-sequence cap, 45% GPU-memory target, nonstream + SDK stream + raw SSE, wrong-model and over-context negatives. Protocol v3 adds exact request/byte replay, complete lifecycle validation and owned process-group teardown. These limits are not a four-user throughput guarantee.


## 0. A byte chunk is not a token or an SSE event — CPU only

The fixture request literally says “Say café ☕ briefly.” Its response, usage counts and arrival times are **invented illustrations**. No tokenizer, model, server or network executes. UTF-8 sequences and CRLF delimiters cross byte-chunk boundaries. Comments do not create data events. Multiline data joins with newline; an unfinished event at EOF is not dispatched.

Predict: why can a role event arrive at 0.03 s but first content at 0.10 s? Why do finish_reason and [DONE] establish different boundaries? Why does a usage-only frame have no choices? Neither delta length nor chunk count gives generated tokens.


In [ ]:
parser_namespace={'__name__':'topic04_cpu_parser'}
exec("\"\"\"Bounded UTF-8/SSE decoder + single-choice chat lifecycle, standard library only.\n\nFraming follows WHATWG event streams (CR/LF/CRLF, comments, multiline data).\nChat completion is a separate, deliberately strict course contract, not all SSE.\n\"\"\"\nimport codecs\nimport json\nimport math\n\n\nclass SSEProtocolError(ValueError):\n    pass\n\n\nclass SSEDecoder:\n    def __init__(self, max_bytes=262144, max_events=4096):\n        self.decoder = codecs.getincrementaldecoder('utf-8-sig')('strict')\n        self.line = ''; self.data = []; self.event = ''; self.after_cr = False\n        self.bytes = 0; self.count = 0; self.max_bytes = max_bytes; self.max_events = max_events\n        self.closed = False\n\n    def _line(self):\n        line, self.line = self.line, ''\n        if not line:\n            if not self.data:\n                self.event = ''; return None\n            event = dict(event=self.event or 'message', data='\\n'.join(self.data))\n            self.data = []; self.event = ''; self.count += 1\n            if self.count > self.max_events: raise SSEProtocolError('event limit exceeded')\n            return event\n        if line.startswith(':'): return None\n        field, sep, value = line.partition(':')\n        if value.startswith(' '): value = value[1:]\n        if field == 'data': self.data.append(value)\n        elif field == 'event': self.event = value\n        # id/retry/unknown fields do not alter this finite HTTP replay client.\n        return None\n\n    def feed(self, raw):\n        if self.closed: raise SSEProtocolError('decoder already closed')\n        if not isinstance(raw,bytes): raise TypeError('feed requires bytes, not decoded strings')\n        self.bytes += len(raw)\n        if self.bytes > self.max_bytes: raise SSEProtocolError('byte limit exceeded')\n        text = self.decoder.decode(raw)\n        events = []\n        for char in text:\n            if self.after_cr:\n                self.after_cr = False\n                if char == '\\n': continue\n            if char in '\\r\\n':\n                event = self._line()\n                if event is not None: events.append(event)\n                self.after_cr = char == '\\r'\n            else: self.line += char\n        return events\n\n    def finish(self):\n        self.decoder.decode(b'',final=True)  # Incomplete UTF-8 is a protocol error.\n        self.closed = True\n        # WHATWG does not dispatch an unfinished event at EOF.\n        return bool(self.line or self.data)\n\n\nclass ChatStream:\n    \"\"\"No reconnect/retry: partial text is evidence, never a completed answer.\"\"\"\n    def __init__(self):\n        self.decoder = SSEDecoder(); self.events = []; self.content = ''\n        self.finish_reason = None; self.usage = None; self.done = False\n        self.first_event_s = None; self.first_content_s = None; self.last_s = 0.\n        self.error = None; self.closed = False; self.cancelled = False; self.unfinished_frame = False\n\n    def _event(self,event,at_s):\n        if self.done: raise SSEProtocolError('data after DONE')\n        if event['event'] != 'message': raise SSEProtocolError('only message events supported')\n        if self.first_event_s is None: self.first_event_s = at_s\n        body=event['data']\n        row=dict(index=len(self.events),at_s=at_s,event=event['event'],data=body)\n        self.events.append(row)\n        if body == '[DONE]':\n            if self.finish_reason is None: raise SSEProtocolError('DONE without finish_reason')\n            self.done = True; row['kind']='done'; return\n        obj=json.loads(body)\n        if not isinstance(obj,dict): raise SSEProtocolError('JSON object required')\n        if 'error' in obj: raise SSEProtocolError('server error event')\n        choices=obj.get('choices')\n        if not isinstance(choices,list): raise SSEProtocolError('choices list required')\n        usage=obj.get('usage')\n        if usage is not None:\n            if not isinstance(usage,dict): raise SSEProtocolError('invalid usage')\n            for key in ('prompt_tokens','completion_tokens','total_tokens'):\n                if type(usage.get(key)) is not int or usage[key]<0: raise SSEProtocolError('invalid usage counts')\n            if usage['total_tokens'] != usage['prompt_tokens']+usage['completion_tokens']:\n                raise SSEProtocolError('usage total mismatch')\n            self.usage=usage\n        if not choices:\n            if usage is None: raise SSEProtocolError('empty choices without usage')\n            row['kind']='usage'; return\n        if len(choices)!=1 or not isinstance(choices[0],dict) or type(choices[0].get('index')) is not int or choices[0]['index']!=0:\n            raise SSEProtocolError('only object choice zero supported')\n        choice=choices[0]; delta=choice.get('delta',{})\n        if not isinstance(delta,dict): raise SSEProtocolError('delta object required')\n        if any(key not in ('role','content') and value is not None for key,value in delta.items()):\n            raise SSEProtocolError('unsupported non-text delta')\n        if delta.get('role') not in (None,'assistant'): raise SSEProtocolError('unsupported role')\n        text=delta.get('content')\n        if text is not None and not isinstance(text,str): raise SSEProtocolError('content must be text or null')\n        if text:\n            if self.finish_reason is not None: raise SSEProtocolError('content after finish')\n            if self.first_content_s is None: self.first_content_s=at_s\n            self.content += text\n        finish=choice.get('finish_reason')\n        if finish is not None:\n            if finish not in ('stop','length') or self.finish_reason is not None:\n                raise SSEProtocolError('invalid or duplicate finish')\n            self.finish_reason=finish\n        row['kind']='finish' if finish is not None else ('content' if text else 'role_or_empty')\n\n    def feed(self,raw,at_s):\n        try:\n            if self.closed: raise SSEProtocolError('stream already closed')\n            if self.error: raise SSEProtocolError('stream already failed')\n            if type(at_s) not in (int,float) or not math.isfinite(at_s) or at_s<self.last_s:\n                raise SSEProtocolError('invalid or nonmonotonic arrival clock')\n            self.last_s=at_s\n            for event in self.decoder.feed(raw): self._event(event,at_s)\n        except (ValueError,TypeError,UnicodeError) as exc:\n            self.error=type(exc).__name__ + ': ' + str(exc)\n            raise\n\n    def finish(self,cancelled=False,transport_error=None):\n        if self.closed: raise SSEProtocolError('stream already closed')\n        self.closed=True; self.cancelled=cancelled\n        try: self.unfinished_frame=self.decoder.finish()\n        except UnicodeError as exc: self.error=type(exc).__name__ + ': incomplete UTF-8'\n        if transport_error: self.error=str(transport_error)\n        if self.error: status='failed'\n        elif cancelled: status='cancelled'\n        elif self.done and self.finish_reason is not None and not self.unfinished_frame: status='complete'\n        else: status='partial'\n        return dict(status=status,content=self.content,finish_reason=self.finish_reason,done=self.done,\n                    usage=self.usage,first_event_s=self.first_event_s,first_content_s=self.first_content_s,\n                    wall_s=self.last_s,byte_count=self.decoder.bytes,event_count=len(self.events),\n                    events=self.events,error=self.error,unfinished_frame=self.unfinished_frame,\n                    clock='client chunk-arrival wall; not token emission or GPU time')\n\n\ndef synthetic_fixture():\n    def chunk(delta,finish=None):\n        return dict(id='illustrative',object='chat.completion.chunk',model='topic04',\n                    choices=[dict(index=0,delta=delta,finish_reason=finish)])\n    payload=dict(model='topic04',messages=[dict(role='user',content='Say café ☕ briefly.')],\n                 max_tokens=16,temperature=0,stream=True,stream_options=dict(include_usage=True))\n    bodies=[chunk(dict(role='assistant',content='')),chunk(dict(content='Café ')),\n            chunk(dict(content='☕.')),chunk({},'stop'),\n            dict(choices=[],usage=dict(prompt_tokens=13,completion_tokens=4,total_tokens=17))]\n    # The synthetic usage counts are invented, not tokenizer measurements.\n    frames=[': heartbeat\\r\\n\\r\\n']+['data: '+json.dumps(b,ensure_ascii=False,separators=(',',':'))+'\\r\\n\\r\\n' for b in bodies]+['data: [DONE]\\r\\n\\r\\n']\n    arrival=[.01,.03,.10,.14,.15,.16,.17]\n    chunks=[]\n    for text,at_s in zip(frames,arrival):\n        raw=text.encode('utf-8')\n        # Always split CRLF; split UTF-8 continuation sequences when present.\n        cuts=sorted(set([len(raw)-3]+[i+1 for i,b in enumerate(raw) if b in (0xc3,0xe2)]))\n        previous=0\n        for end in cuts+[len(raw)]:\n            chunks.append(dict(at_s=at_s,hex=raw[previous:end].hex())); previous=end\n    stream=ChatStream()\n    for c in chunks: stream.feed(bytes.fromhex(c['hex']),c['at_s'])\n    complete=stream.finish()\n    prefix=ChatStream()\n    for c in chunks:\n        if c['at_s']>.10: break\n        prefix.feed(bytes.fromhex(c['hex']),c['at_s'])\n    partial=prefix.finish()\n    cancelled=ChatStream()\n    for c in chunks:\n        if c['at_s']>.10: break\n        cancelled.feed(bytes.fromhex(c['hex']),c['at_s'])\n    return dict(schema='topic04-sse-fixture-v1',kind='illustrative bytes, clocks and usage; no server or tokenizer executed',\n                request=payload,frames=frames,chunks=chunks,complete=complete,partial=partial,\n                cancelled=cancelled.finish(cancelled=True),source='https://html.spec.whatwg.org/multipage/server-sent-events.html')\n",parser_namespace)
FIXTURE=parser_namespace['synthetic_fixture']()
print(FIXTURE['kind'])
print('Literal request:',FIXTURE['request'])
stream=parser_namespace['ChatStream']()
for index,chunk in enumerate(FIXTURE['chunks']):
    raw=bytes.fromhex(chunk['hex'])
    print('byte chunk',index,'at',chunk['at_s'],'s:',repr(raw))
    stream.feed(raw,chunk['at_s'])
COMPLETE=stream.finish()
for event in COMPLETE['events']: print(event['index'],event['at_s'],event['kind'],event['data'])
print('Parsed result:',{k:v for k,v in COMPLETE.items() if k!='events'})
assert COMPLETE==FIXTURE['complete']
assert COMPLETE['first_event_s']==.03 and COMPLETE['first_content_s']==.10


### Failure drill: preserve partial content, never call it complete

The same prefix can end because the peer disconnected, the client cancelled or the transport failed. Keep those classifications separate. A valid finish frame without [DONE] still does not satisfy this course's raw completion contract. The parser rejects invalid JSON/UTF-8, data after [DONE], duplicate finish and bounded-byte/event overflow; the helper intentionally supports only one choice at index 0.

Synthetic cancellation tests client bookkeeping only. It does **not** prove that an actual engine reclaimed KV, stopped GPU work or suppressed billing. Those need a separate server-side reclamation experiment.


In [ ]:
for label in ('partial','cancelled'):
    result=FIXTURE[label]
    print(label,result['status'],repr(result['content']),'finish:',result['finish_reason'],'DONE:',result['done'])
assert FIXTURE['partial']['status']=='partial'
assert FIXTURE['cancelled']['status']=='cancelled'
broken=parser_namespace['ChatStream']()
try: broken.feed(b'data: [DONE]\n\n',.1)
except ValueError as exc: print('Expected malformed-lifecycle rejection:',type(exc).__name__)
assert broken.finish()['status']=='failed'
# Replay one byte at a time: decoding must not depend on network segmentation.
resegmented=parser_namespace['ChatStream']()
wire=''.join(FIXTURE['frames']).encode('utf-8')
for index,value in enumerate(wire): resegmented.feed(bytes([value]),index/1000)
assert resegmented.finish()['content']==COMPLETE['content']


## 1. Recorded real HTTP/SSE request — not the synthetic fixture

The raw live probe says “Say hello briefly.” The separate SDK/nonstream prompt asks for one short sentence explaining an API server. Inspect the rendered chat and actual input IDs for each; never attach the synthetic café IDs or times to this request.

The byte transcript is the exact finite response observed by the instructor HTTP client, with per-chunk arrival clocks. Replaying it sends no request. Request IDs inside response JSON are public teaching-response identifiers, not authentication secrets. Timings include client/server/transport effects and startup state; they are not pure GPU decode time or a steady-state latency claim.


In [ ]:
import json
RECORDED_SERVICE=json.loads("{\n  \"contract\": \"topic-04-api-smoke-v3\",\n  \"status\": \"complete\",\n  \"model_id\": \"Qwen/Qwen2.5-0.5B-Instruct\",\n  \"revision\": \"7ae557604adf67be50417f59c2c2f167def9a775\",\n  \"served_name\": \"topic04\",\n  \"utc_started\": \"2026-10-01T22:05:29.847952+00:00\",\n  \"vllm\": \"0.29.0\",\n  \"openai_sdk\": \"3.20.0\",\n  \"httpx\": \"0.28.1\",\n  \"transformers\": \"5.17.0\",\n  \"torch\": \"2.13.0+cu130\",\n  \"cuda\": \"13.0\",\n  \"gpu\": \"NVIDIA GeForce RTX 5070 Ti\",\n  \"launch_args\": [\n    \"serve\",\n    \"<pinned-local-snapshot>\",\n    \"--served-model-name\",\n    \"topic04\",\n    \"--host\",\n    \"127.0.0.1\",\n    \"--port\",\n    \"8000\",\n    \"--dtype\",\n    \"float16\",\n    \"--max-model-len\",\n    \"2048\",\n    \"--max-num-seqs\",\n    \"4\",\n    \"--gpu-memory-utilization\",\n    \"0.45\",\n    \"--enforce-eager\"\n  ],\n  \"wsl_v1_runner_fallback\": true,\n  \"resource_samples\": [\n    {\n      \"available_ram_gib\": 68.89566421508789,\n      \"device_used_mib\": 0\n    },\n    {\n      \"available_ram_gib\": 68.76938247680664,\n      \"device_used_mib\": 0\n    }\n  ],\n  \"request_prompt\": {\n    \"messages\": [\n      {\n        \"role\": \"user\",\n        \"content\": \"Reply with one short sentence explaining what an API server does.\"\n      }\n    ],\n    \"rendered_chat\": \"<|im_start|>system\\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\\n<|im_start|>user\\nReply with one short sentence explaining what an API server does.<|im_end|>\\n<|im_start|>assistant\\n\",\n    \"input_token_ids\": [\n      151644,\n      8948,\n      198,\n      2610,\n      525,\n      1207,\n      16948,\n      11,\n      3465,\n      553,\n      54364,\n      14817,\n      13,\n      1446,\n      525,\n      264,\n      10950,\n      17847,\n      13,\n      151645,\n      198,\n      151644,\n      872,\n      198,\n      20841,\n      448,\n      825,\n      2805,\n      11652,\n      25021,\n      1128,\n      458,\n      5333,\n      3538,\n      1558,\n      13,\n      151645,\n      198,\n      151644,\n      77091,\n      198\n    ],\n    \"prompt_tokens\": 41,\n    \"input_token_pieces\": [\n      \"<|im_start|>\",\n      \"system\",\n      \"Ċ\",\n      \"You\",\n      \"Ġare\",\n      \"ĠQ\",\n      \"wen\",\n      \",\",\n      \"Ġcreated\",\n      \"Ġby\",\n      \"ĠAlibaba\",\n      \"ĠCloud\",\n      \".\",\n      \"ĠYou\",\n      \"Ġare\",\n      \"Ġa\",\n      \"Ġhelpful\",\n      \"Ġassistant\",\n      \".\",\n      \"<|im_end|>\",\n      \"Ċ\",\n      \"<|im_start|>\",\n      \"user\",\n      \"Ċ\",\n      \"Reply\",\n      \"Ġwith\",\n      \"Ġone\",\n      \"Ġshort\",\n      \"Ġsentence\",\n      \"Ġexplaining\",\n      \"Ġwhat\",\n      \"Ġan\",\n      \"ĠAPI\",\n      \"Ġserver\",\n      \"Ġdoes\",\n      \".\",\n      \"<|im_end|>\",\n      \"Ċ\",\n      \"<|im_start|>\",\n      \"assistant\",\n      \"Ċ\"\n    ]\n  },\n  \"raw_prompt\": {\n    \"messages\": [\n      {\n        \"role\": \"user\",\n        \"content\": \"Say hello briefly.\"\n      }\n    ],\n    \"rendered_chat\": \"<|im_start|>system\\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\\n<|im_start|>user\\nSay hello briefly.<|im_end|>\\n<|im_start|>assistant\\n\",\n    \"input_token_ids\": [\n      151644,\n      8948,\n      198,\n      2610,\n      525,\n      1207,\n      16948,\n      11,\n      3465,\n      553,\n      54364,\n      14817,\n      13,\n      1446,\n      525,\n      264,\n      10950,\n      17847,\n      13,\n      151645,\n      198,\n      151644,\n      872,\n      198,\n      45764,\n      23811,\n      26753,\n      13,\n      151645,\n      198,\n      151644,\n      77091,\n      198\n    ],\n    \"prompt_tokens\": 33,\n    \"input_token_pieces\": [\n      \"<|im_start|>\",\n      \"system\",\n      \"Ċ\",\n      \"You\",\n      \"Ġare\",\n      \"ĠQ\",\n      \"wen\",\n      \",\",\n      \"Ġcreated\",\n      \"Ġby\",\n      \"ĠAlibaba\",\n      \"ĠCloud\",\n      \".\",\n      \"ĠYou\",\n      \"Ġare\",\n      \"Ġa\",\n      \"Ġhelpful\",\n      \"Ġassistant\",\n      \".\",\n      \"<|im_end|>\",\n      \"Ċ\",\n      \"<|im_start|>\",\n      \"user\",\n      \"Ċ\",\n      \"Say\",\n      \"Ġhello\",\n      \"Ġbriefly\",\n      \".\",\n      \"<|im_end|>\",\n      \"Ċ\",\n      \"<|im_start|>\",\n      \"assistant\",\n      \"Ċ\"\n    ]\n  },\n  \"limits\": {\n    \"max_model_len\": 2048,\n    \"max_num_seqs\": 4,\n    \"gpu_memory_utilization\": 0.45\n  },\n  \"limitations\": [\n    \"single sequential API smoke, not throughput benchmark\",\n    \"first call may include lazy runtime initialization\",\n    \"TCP/HTTP chunk boundaries are not generated token boundaries\",\n    \"client disconnect reclamation not measured\"\n  ],\n  \"advertised_models\": [\n    \"topic04\"\n  ],\n  \"startup_s\": 24.456222492999586,\n  \"nonstream\": {\n    \"request\": {\n      \"model\": \"topic04\",\n      \"messages\": [\n        {\n          \"role\": \"user\",\n          \"content\": \"Reply with one short sentence explaining what an API server does.\"\n        }\n      ],\n      \"max_tokens\": 48,\n      \"temperature\": 0\n    },\n    \"wall_s\": 0.8830939609997586,\n    \"finish_reason\": \"stop\",\n    \"usage\": {\n      \"completion_tokens\": 29,\n      \"prompt_tokens\": 41,\n      \"total_tokens\": 70,\n      \"completion_tokens_details\": null,\n      \"prompt_tokens_details\": null\n    },\n    \"content\": \"An API server is a software application that acts as the gateway for clients to communicate with another system or service over an internet protocol (HTTP).\"\n  },\n  \"sdk_stream\": {\n    \"first_content_s\": 0.06405494099999487,\n    \"wall_s\": 0.5953651939998963,\n    \"finish_reason\": \"stop\",\n    \"content\": \"An API server is a software application that acts as the gateway for clients to communicate with another system or service over an internet protocol (HTTP).\",\n    \"sdk_chunks\": 30\n  },\n  \"raw_sse\": {\n    \"status\": \"complete\",\n    \"content\": \"Hello! How can I assist you today?\",\n    \"finish_reason\": \"stop\",\n    \"done\": true,\n    \"usage\": {\n      \"prompt_tokens\": 33,\n      \"total_tokens\": 43,\n      \"completion_tokens\": 10\n    },\n    \"first_event_s\": 0.05866083799992339,\n    \"first_content_s\": 0.05872152499978256,\n    \"wall_s\": 0.17361840400008077,\n    \"byte_count\": 2718,\n    \"event_count\": 13,\n    \"events\": [\n      {\n        \"index\": 0,\n        \"at_s\": 0.05866083799992339,\n        \"event\": \"message\",\n        \"data\": \"{\\\"id\\\":\\\"chatcmpl-ba4deba68c98bed8\\\",\\\"object\\\":\\\"chat.completion.chunk\\\",\\\"created\\\":1790892355,\\\"model\\\":\\\"topic04\\\",\\\"choices\\\":[{\\\"index\\\":0,\\\"delta\\\":{\\\"role\\\":\\\"assistant\\\",\\\"content\\\":\\\"\\\"},\\\"logprobs\\\":null,\\\"finish_reason\\\":null}],\\\"prompt_token_ids\\\":null,\\\"prompt_text\\\":null}\",\n        \"kind\": \"role_or_empty\"\n      },\n      {\n        \"index\": 1,\n        \"at_s\": 0.05872152499978256,\n        \"event\": \"message\",\n        \"data\": \"{\\\"id\\\":\\\"chatcmpl-ba4deba68c98bed8\\\",\\\"object\\\":\\\"chat.completion.chunk\\\",\\\"created\\\":1790892355,\\\"model\\\":\\\"topic04\\\",\\\"choices\\\":[{\\\"index\\\":0,\\\"delta\\\":{\\\"content\\\":\\\"Hello\\\"},\\\"logprobs\\\":null,\\\"finish_reason\\\":null,\\\"token_ids\\\":null}]}\",\n        \"kind\": \"content\"\n      },\n      {\n        \"index\": 2,\n        \"at_s\": 0.0673291999992216,\n        \"event\": \"message\",\n        \"data\": \"{\\\"id\\\":\\\"chatcmpl-ba4deba68c98bed8\\\",\\\"object\\\":\\\"chat.completion.chunk\\\",\\\"created\\\":1790892355,\\\"model\\\":\\\"topic04\\\",\\\"choices\\\":[{\\\"index\\\":0,\\\"delta\\\":{\\\"content\\\":\\\"!\\\"},\\\"logprobs\\\":null,\\\"finish_reason\\\":null,\\\"token_ids\\\":null}]}\",\n        \"kind\": \"content\"\n      },\n      {\n        \"index\": 3,\n        \"at_s\": 0.07651217199963867,\n        \"event\": \"message\",\n        \"data\": \"{\\\"id\\\":\\\"chatcmpl-ba4deba68c98bed8\\\",\\\"object\\\":\\\"chat.completion.chunk\\\",\\\"created\\\":1790892355,\\\"model\\\":\\\"topic04\\\",\\\"choices\\\":[{\\\"index\\\":0,\\\"delta\\\":{\\\"content\\\":\\\" How\\\"},\\\"logprobs\\\":null,\\\"finish_reason\\\":null,\\\"token_ids\\\":null}]}\",\n        \"kind\": \"content\"\n      },\n      {\n        \"index\": 4,\n        \"at_s\": 0.08649925200006692,\n        \"event\": \"message\",\n        \"data\": \"{\\\"id\\\":\\\"chatcmpl-ba4deba68c98bed8\\\",\\\"object\\\":\\\"chat.completion.chunk\\\",\\\"created\\\":1790892355,\\\"model\\\":\\\"topic04\\\",\\\"choices\\\":[{\\\"index\\\":0,\\\"delta\\\":{\\\"content\\\":\\\" can\\\"},\\\"logprobs\\\":null,\\\"finish_reason\\\":null,\\\"token_ids\\\":null}]}\",\n        \"kind\": \"content\"\n      },\n      {\n        \"index\": 5,\n        \"at_s\": 0.09595841199916322,\n        \"event\": \"message\",\n        \"data\": \"{\\\"id\\\":\\\"chatcmpl-ba4deba68c98bed8\\\",\\\"object\\\":\\\"chat.completion.chunk\\\",\\\"created\\\":1790892355,\\\"model\\\":\\\"topic04\\\",\\\"choices\\\":[{\\\"index\\\":0,\\\"delta\\\":{\\\"content\\\":\\\" I\\\"},\\\"logprobs\\\":null,\\\"finish_reason\\\":null,\\\"token_ids\\\":null}]}\",\n        \"kind\": \"content\"\n      },\n      {\n        \"index\": 6,\n        \"at_s\": 0.10532649800006766,\n        \"event\": \"message\",\n        \"data\": \"{\\\"id\\\":\\\"chatcmpl-ba4deba68c98bed8\\\",\\\"object\\\":\\\"chat.completion.chunk\\\",\\\"created\\\":1790892355,\\\"model\\\":\\\"topic04\\\",\\\"choices\\\":[{\\\"index\\\":0,\\\"delta\\\":{\\\"content\\\":\\\" assist\\\"},\\\"logprobs\\\":null,\\\"finish_reason\\\":null,\\\"token_ids\\\":null}]}\",\n        \"kind\": \"content\"\n      },\n      {\n        \"index\": 7,\n        \"at_s\": 0.11451939499966102,\n        \"event\": \"message\",\n        \"data\": \"{\\\"id\\\":\\\"chatcmpl-ba4deba68c98bed8\\\",\\\"object\\\":\\\"chat.completion.chunk\\\",\\\"created\\\":1790892355,\\\"model\\\":\\\"topic04\\\",\\\"choices\\\":[{\\\"index\\\":0,\\\"delta\\\":{\\\"content\\\":\\\" you\\\"},\\\"logprobs\\\":null,\\\"finish_reason\\\":null,\\\"token_ids\\\":null}]}\",\n        \"kind\": \"content\"\n      },\n      {\n        \"index\": 8,\n        \"at_s\": 0.12420863999977882,\n        \"event\": \"message\",\n        \"data\": \"{\\\"id\\\":\\\"chatcmpl-ba4deba68c98bed8\\\",\\\"object\\\":\\\"chat.completion.chunk\\\",\\\"created\\\":1790892355,\\\"model\\\":\\\"topic04\\\",\\\"choices\\\":[{\\\"index\\\":0,\\\"delta\\\":{\\\"content\\\":\\\" today\\\"},\\\"logprobs\\\":null,\\\"finish_reason\\\":null,\\\"token_ids\\\":null}]}\",\n        \"kind\": \"content\"\n      },\n      {\n        \"index\": 9,\n        \"at_s\": 0.14300291300060053,\n        \"event\": \"message\",\n        \"data\": \"{\\\"id\\\":\\\"chatcmpl-ba4deba68c98bed8\\\",\\\"object\\\":\\\"chat.completion.chunk\\\",\\\"created\\\":1790892355,\\\"model\\\":\\\"topic04\\\",\\\"choices\\\":[{\\\"index\\\":0,\\\"delta\\\":{\\\"content\\\":\\\"?\\\"},\\\"logprobs\\\":null,\\\"finish_reason\\\":null,\\\"token_ids\\\":null}]}\",\n        \"kind\": \"content\"\n      },\n      {\n        \"index\": 10,\n        \"at_s\": 0.17354318899924692,\n        \"event\": \"message\",\n        \"data\": \"{\\\"id\\\":\\\"chatcmpl-ba4deba68c98bed8\\\",\\\"object\\\":\\\"chat.completion.chunk\\\",\\\"created\\\":1790892355,\\\"model\\\":\\\"topic04\\\",\\\"choices\\\":[{\\\"index\\\":0,\\\"delta\\\":{\\\"content\\\":\\\"\\\"},\\\"logprobs\\\":null,\\\"finish_reason\\\":\\\"stop\\\",\\\"stop_reason\\\":null,\\\"token_ids\\\":null}]}\",\n        \"kind\": \"finish\"\n      },\n      {\n        \"index\": 11,\n        \"at_s\": 0.17358486999910383,\n        \"event\": \"message\",\n        \"data\": \"{\\\"id\\\":\\\"chatcmpl-ba4deba68c98bed8\\\",\\\"object\\\":\\\"chat.completion.chunk\\\",\\\"created\\\":1790892355,\\\"model\\\":\\\"topic04\\\",\\\"choices\\\":[],\\\"usage\\\":{\\\"prompt_tokens\\\":33,\\\"total_tokens\\\":43,\\\"completion_tokens\\\":10},\\\"system_fingerprint\\\":\\\"vllm-0.29.0-nohash\\\"}\",\n        \"kind\": \"usage\"\n      },\n      {\n        \"index\": 12,\n        \"at_s\": 0.17361840400008077,\n        \"event\": \"message\",\n        \"data\": \"[DONE]\",\n        \"kind\": \"done\"\n      }\n    ],\n    \"error\": null,\n    \"unfinished_frame\": false,\n    \"clock\": \"client chunk-arrival wall; not token emission or GPU time\",\n    \"request\": {\n      \"model\": \"topic04\",\n      \"messages\": [\n        {\n          \"role\": \"user\",\n          \"content\": \"Say hello briefly.\"\n        }\n      ],\n      \"max_tokens\": 16,\n      \"temperature\": 0,\n      \"stream\": true,\n      \"stream_options\": {\n        \"include_usage\": true\n      }\n    },\n    \"http_status\": 200,\n    \"content_type\": \"text/event-stream\",\n    \"chunks\": [\n      {\n        \"at_s\": 0.05866083799992339,\n        \"hex\": \"646174613a207b226964223a2263686174636d706c2d62613464656261363863393862656438222c226f626a656374223a22636861742e636f6d706c6574696f6e2e6368756e6b222c2263726561746564223a313739303839323335352c226d6f64656c223a22746f7069633034222c2263686f69636573223a5b7b22696e646578223a302c2264656c7461223a7b22726f6c65223a22617373697374616e74222c22636f6e74656e74223a22227d2c226c6f6770726f6273223a6e756c6c2c2266696e6973685f726561736f6e223a6e756c6c7d5d2c2270726f6d70745f746f6b656e5f696473223a6e756c6c2c2270726f6d70745f74657874223a6e756c6c7d0a0a\"\n      },\n      {\n        \"at_s\": 0.05872152499978256,\n        \"hex\": \"646174613a207b226964223a2263686174636d706c2d62613464656261363863393862656438222c226f626a656374223a22636861742e636f6d706c6574696f6e2e6368756e6b222c2263726561746564223a313739303839323335352c226d6f64656c223a22746f7069633034222c2263686f69636573223a5b7b22696e646578223a302c2264656c7461223a7b22636f6e74656e74223a2248656c6c6f227d2c226c6f6770726f6273223a6e756c6c2c2266696e6973685f726561736f6e223a6e756c6c2c22746f6b656e5f696473223a6e756c6c7d5d7d0a0a\"\n      },\n      {\n        \"at_s\": 0.0673291999992216,\n        \"hex\": \"646174613a207b226964223a2263686174636d706c2d62613464656261363863393862656438222c226f626a656374223a22636861742e636f6d706c6574696f6e2e6368756e6b222c2263726561746564223a313739303839323335352c226d6f64656c223a22746f7069633034222c2263686f69636573223a5b7b22696e646578223a302c2264656c7461223a7b22636f6e74656e74223a2221227d2c226c6f6770726f6273223a6e756c6c2c2266696e6973685f726561736f6e223a6e756c6c2c22746f6b656e5f696473223a6e756c6c7d5d7d0a0a\"\n      },\n      {\n        \"at_s\": 0.07651217199963867,\n        \"hex\": \"646174613a207b226964223a2263686174636d706c2d62613464656261363863393862656438222c226f626a656374223a22636861742e636f6d706c6574696f6e2e6368756e6b222c2263726561746564223a313739303839323335352c226d6f64656c223a22746f7069633034222c2263686f69636573223a5b7b22696e646578223a302c2264656c7461223a7b22636f6e74656e74223a2220486f77227d2c226c6f6770726f6273223a6e756c6c2c2266696e6973685f726561736f6e223a6e756c6c2c22746f6b656e5f696473223a6e756c6c7d5d7d0a0a\"\n      },\n      {\n        \"at_s\": 0.08649925200006692,\n        \"hex\": \"646174613a207b226964223a2263686174636d706c2d62613464656261363863393862656438222c226f626a656374223a22636861742e636f6d706c6574696f6e2e6368756e6b222c2263726561746564223a313739303839323335352c226d6f64656c223a22746f7069633034222c2263686f69636573223a5b7b22696e646578223a302c2264656c7461223a7b22636f6e74656e74223a222063616e227d2c226c6f6770726f6273223a6e756c6c2c2266696e6973685f726561736f6e223a6e756c6c2c22746f6b656e5f696473223a6e756c6c7d5d7d0a0a\"\n      },\n      {\n        \"at_s\": 0.09595841199916322,\n        \"hex\": \"646174613a207b226964223a2263686174636d706c2d62613464656261363863393862656438222c226f626a656374223a22636861742e636f6d706c6574696f6e2e6368756e6b222c2263726561746564223a313739303839323335352c226d6f64656c223a22746f7069633034222c2263686f69636573223a5b7b22696e646578223a302c2264656c7461223a7b22636f6e74656e74223a222049227d2c226c6f6770726f6273223a6e756c6c2c2266696e6973685f726561736f6e223a6e756c6c2c22746f6b656e5f696473223a6e756c6c7d5d7d0a0a\"\n      },\n      {\n        \"at_s\": 0.10532649800006766,\n        \"hex\": \"646174613a207b226964223a2263686174636d706c2d62613464656261363863393862656438222c226f626a656374223a22636861742e636f6d706c6574696f6e2e6368756e6b222c2263726561746564223a313739303839323335352c226d6f64656c223a22746f7069633034222c2263686f69636573223a5b7b22696e646578223a302c2264656c7461223a7b22636f6e74656e74223a2220617373697374227d2c226c6f6770726f6273223a6e756c6c2c2266696e6973685f726561736f6e223a6e756c6c2c22746f6b656e5f696473223a6e756c6c7d5d7d0a0a\"\n      },\n      {\n        \"at_s\": 0.11451939499966102,\n        \"hex\": \"646174613a207b226964223a2263686174636d706c2d62613464656261363863393862656438222c226f626a656374223a22636861742e636f6d706c6574696f6e2e6368756e6b222c2263726561746564223a313739303839323335352c226d6f64656c223a22746f7069633034222c2263686f69636573223a5b7b22696e646578223a302c2264656c7461223a7b22636f6e74656e74223a2220796f75227d2c226c6f6770726f6273223a6e756c6c2c2266696e6973685f726561736f6e223a6e756c6c2c22746f6b656e5f696473223a6e756c6c7d5d7d0a0a\"\n      },\n      {\n        \"at_s\": 0.12420863999977882,\n        \"hex\": \"646174613a207b226964223a2263686174636d706c2d62613464656261363863393862656438222c226f626a656374223a22636861742e636f6d706c6574696f6e2e6368756e6b222c2263726561746564223a313739303839323335352c226d6f64656c223a22746f7069633034222c2263686f69636573223a5b7b22696e646578223a302c2264656c7461223a7b22636f6e74656e74223a2220746f646179227d2c226c6f6770726f6273223a6e756c6c2c2266696e6973685f726561736f6e223a6e756c6c2c22746f6b656e5f696473223a6e756c6c7d5d7d0a0a\"\n      },\n      {\n        \"at_s\": 0.14300291300060053,\n        \"hex\": \"646174613a207b226964223a2263686174636d706c2d62613464656261363863393862656438222c226f626a656374223a22636861742e636f6d706c6574696f6e2e6368756e6b222c2263726561746564223a313739303839323335352c226d6f64656c223a22746f7069633034222c2263686f69636573223a5b7b22696e646578223a302c2264656c7461223a7b22636f6e74656e74223a223f227d2c226c6f6770726f6273223a6e756c6c2c2266696e6973685f726561736f6e223a6e756c6c2c22746f6b656e5f696473223a6e756c6c7d5d7d0a0a\"\n      },\n      {\n        \"at_s\": 0.17354318899924692,\n        \"hex\": \"646174613a207b226964223a2263686174636d706c2d62613464656261363863393862656438222c226f626a656374223a22636861742e636f6d706c6574696f6e2e6368756e6b222c2263726561746564223a313739303839323335352c226d6f64656c223a22746f7069633034222c2263686f69636573223a5b7b22696e646578223a302c2264656c7461223a7b22636f6e74656e74223a22227d2c226c6f6770726f6273223a6e756c6c2c2266696e6973685f726561736f6e223a2273746f70222c2273746f705f726561736f6e223a6e756c6c2c22746f6b656e5f696473223a6e756c6c7d5d7d0a0a\"\n      },\n      {\n        \"at_s\": 0.17358486999910383,\n        \"hex\": \"646174613a207b226964223a2263686174636d706c2d62613464656261363863393862656438222c226f626a656374223a22636861742e636f6d706c6574696f6e2e6368756e6b222c2263726561746564223a313739303839323335352c226d6f64656c223a22746f7069633034222c2263686f69636573223a5b5d2c227573616765223a7b2270726f6d70745f746f6b656e73223a33332c22746f74616c5f746f6b656e73223a34332c22636f6d706c6574696f6e5f746f6b656e73223a31307d2c2273797374656d5f66696e6765727072696e74223a22766c6c6d2d302e32392e302d6e6f68617368227d0a0a\"\n      },\n      {\n        \"at_s\": 0.17361840400008077,\n        \"hex\": \"646174613a205b444f4e455d0a0a\"\n      }\n    ]\n  },\n  \"negative\": {\n    \"wrong_model\": {\n      \"status\": 404\n    },\n    \"over_context\": {\n      \"status\": 400,\n      \"rendered_prompt_tokens\": 2329\n    }\n  },\n  \"teardown\": {\n    \"owned_process_group_stopped\": true,\n    \"server_returncode\": 0\n  },\n  \"utc_finished\": \"2026-10-01T22:05:56.753374+00:00\"\n}\n")
for key in ('request_prompt','raw_prompt'):
    prompt=RECORDED_SERVICE[key]
    print('\nRECORDED',key,prompt['messages'])
    print('Rendered chat:',repr(prompt['rendered_chat']))
    print('Actual input IDs:',prompt['input_token_ids'],'count:',prompt['prompt_tokens'])
actual=parser_namespace['ChatStream']()
for chunk in RECORDED_SERVICE['raw_sse']['chunks']:
    actual.feed(bytes.fromhex(chunk['hex']),chunk['at_s'])
REPLAY=actual.finish()
print('Real raw request:',RECORDED_SERVICE['raw_sse']['request'])
for event in REPLAY['events']: print(event['index'],event['at_s'],event['kind'],event['data'])
print('Content:',repr(REPLAY['content']),'usage:',REPLAY['usage'],'status:',REPLAY['status'])
assert REPLAY['status']=='complete'
assert REPLAY['content']==RECORDED_SERVICE['raw_sse']['content']
print('Negative probes:',RECORDED_SERVICE['negative'])
print('Recorded replay is not evidence that you ran a GPU service.')


## 2. Check the machine and resolve an immutable cache

Use Linux/WSL with an otherwise idle CUDA GPU and at least 24 GiB available RAM. Run one GPU process only; the notebook refuses >12 GiB device use and does not lower these gates automatically. Instructor execution also verifies a 24 GiB/no-swap scope and 900 s external wall timeout. A gate is not an OS sandbox. If your machine fails readiness, report blocked and obtain an appropriate CUDA environment.

The install cell skips an already matching vLLM version. Reuse LSM_MODEL_CACHE on persistent storage; an offline missing snapshot fails without downloading. Stop before proceeding if the installed runtime is incompatible; do not silently substitute a different server or model. WSL records the V1/native sampler fallback required by this host.


In [ ]:
import subprocess,sys
from importlib.metadata import version,PackageNotFoundError
try: matches=version('vllm')=='0.29.0'
except PackageNotFoundError: matches=False
if not matches: subprocess.check_call([sys.executable,'-m','pip','install','vllm==0.29.0'])


In [ ]:
lab_namespace={'__name__':'topic04_student_lab'}
exec("\"\"\"Bounded UTF-8/SSE decoder + single-choice chat lifecycle, standard library only.\n\nFraming follows WHATWG event streams (CR/LF/CRLF, comments, multiline data).\nChat completion is a separate, deliberately strict course contract, not all SSE.\n\"\"\"\nimport codecs\nimport json\nimport math\n\n\nclass SSEProtocolError(ValueError):\n    pass\n\n\nclass SSEDecoder:\n    def __init__(self, max_bytes=262144, max_events=4096):\n        self.decoder = codecs.getincrementaldecoder('utf-8-sig')('strict')\n        self.line = ''; self.data = []; self.event = ''; self.after_cr = False\n        self.bytes = 0; self.count = 0; self.max_bytes = max_bytes; self.max_events = max_events\n        self.closed = False\n\n    def _line(self):\n        line, self.line = self.line, ''\n        if not line:\n            if not self.data:\n                self.event = ''; return None\n            event = dict(event=self.event or 'message', data='\\n'.join(self.data))\n            self.data = []; self.event = ''; self.count += 1\n            if self.count > self.max_events: raise SSEProtocolError('event limit exceeded')\n            return event\n        if line.startswith(':'): return None\n        field, sep, value = line.partition(':')\n        if value.startswith(' '): value = value[1:]\n        if field == 'data': self.data.append(value)\n        elif field == 'event': self.event = value\n        # id/retry/unknown fields do not alter this finite HTTP replay client.\n        return None\n\n    def feed(self, raw):\n        if self.closed: raise SSEProtocolError('decoder already closed')\n        if not isinstance(raw,bytes): raise TypeError('feed requires bytes, not decoded strings')\n        self.bytes += len(raw)\n        if self.bytes > self.max_bytes: raise SSEProtocolError('byte limit exceeded')\n        text = self.decoder.decode(raw)\n        events = []\n        for char in text:\n            if self.after_cr:\n                self.after_cr = False\n                if char == '\\n': continue\n            if char in '\\r\\n':\n                event = self._line()\n                if event is not None: events.append(event)\n                self.after_cr = char == '\\r'\n            else: self.line += char\n        return events\n\n    def finish(self):\n        self.decoder.decode(b'',final=True)  # Incomplete UTF-8 is a protocol error.\n        self.closed = True\n        # WHATWG does not dispatch an unfinished event at EOF.\n        return bool(self.line or self.data)\n\n\nclass ChatStream:\n    \"\"\"No reconnect/retry: partial text is evidence, never a completed answer.\"\"\"\n    def __init__(self):\n        self.decoder = SSEDecoder(); self.events = []; self.content = ''\n        self.finish_reason = None; self.usage = None; self.done = False\n        self.first_event_s = None; self.first_content_s = None; self.last_s = 0.\n        self.error = None; self.closed = False; self.cancelled = False; self.unfinished_frame = False\n\n    def _event(self,event,at_s):\n        if self.done: raise SSEProtocolError('data after DONE')\n        if event['event'] != 'message': raise SSEProtocolError('only message events supported')\n        if self.first_event_s is None: self.first_event_s = at_s\n        body=event['data']\n        row=dict(index=len(self.events),at_s=at_s,event=event['event'],data=body)\n        self.events.append(row)\n        if body == '[DONE]':\n            if self.finish_reason is None: raise SSEProtocolError('DONE without finish_reason')\n            self.done = True; row['kind']='done'; return\n        obj=json.loads(body)\n        if not isinstance(obj,dict): raise SSEProtocolError('JSON object required')\n        if 'error' in obj: raise SSEProtocolError('server error event')\n        choices=obj.get('choices')\n        if not isinstance(choices,list): raise SSEProtocolError('choices list required')\n        usage=obj.get('usage')\n        if usage is not None:\n            if not isinstance(usage,dict): raise SSEProtocolError('invalid usage')\n            for key in ('prompt_tokens','completion_tokens','total_tokens'):\n                if type(usage.get(key)) is not int or usage[key]<0: raise SSEProtocolError('invalid usage counts')\n            if usage['total_tokens'] != usage['prompt_tokens']+usage['completion_tokens']:\n                raise SSEProtocolError('usage total mismatch')\n            self.usage=usage\n        if not choices:\n            if usage is None: raise SSEProtocolError('empty choices without usage')\n            row['kind']='usage'; return\n        if len(choices)!=1 or not isinstance(choices[0],dict) or type(choices[0].get('index')) is not int or choices[0]['index']!=0:\n            raise SSEProtocolError('only object choice zero supported')\n        choice=choices[0]; delta=choice.get('delta',{})\n        if not isinstance(delta,dict): raise SSEProtocolError('delta object required')\n        if any(key not in ('role','content') and value is not None for key,value in delta.items()):\n            raise SSEProtocolError('unsupported non-text delta')\n        if delta.get('role') not in (None,'assistant'): raise SSEProtocolError('unsupported role')\n        text=delta.get('content')\n        if text is not None and not isinstance(text,str): raise SSEProtocolError('content must be text or null')\n        if text:\n            if self.finish_reason is not None: raise SSEProtocolError('content after finish')\n            if self.first_content_s is None: self.first_content_s=at_s\n            self.content += text\n        finish=choice.get('finish_reason')\n        if finish is not None:\n            if finish not in ('stop','length') or self.finish_reason is not None:\n                raise SSEProtocolError('invalid or duplicate finish')\n            self.finish_reason=finish\n        row['kind']='finish' if finish is not None else ('content' if text else 'role_or_empty')\n\n    def feed(self,raw,at_s):\n        try:\n            if self.closed: raise SSEProtocolError('stream already closed')\n            if self.error: raise SSEProtocolError('stream already failed')\n            if type(at_s) not in (int,float) or not math.isfinite(at_s) or at_s<self.last_s:\n                raise SSEProtocolError('invalid or nonmonotonic arrival clock')\n            self.last_s=at_s\n            for event in self.decoder.feed(raw): self._event(event,at_s)\n        except (ValueError,TypeError,UnicodeError) as exc:\n            self.error=type(exc).__name__ + ': ' + str(exc)\n            raise\n\n    def finish(self,cancelled=False,transport_error=None):\n        if self.closed: raise SSEProtocolError('stream already closed')\n        self.closed=True; self.cancelled=cancelled\n        try: self.unfinished_frame=self.decoder.finish()\n        except UnicodeError as exc: self.error=type(exc).__name__ + ': incomplete UTF-8'\n        if transport_error: self.error=str(transport_error)\n        if self.error: status='failed'\n        elif cancelled: status='cancelled'\n        elif self.done and self.finish_reason is not None and not self.unfinished_frame: status='complete'\n        else: status='partial'\n        return dict(status=status,content=self.content,finish_reason=self.finish_reason,done=self.done,\n                    usage=self.usage,first_event_s=self.first_event_s,first_content_s=self.first_content_s,\n                    wall_s=self.last_s,byte_count=self.decoder.bytes,event_count=len(self.events),\n                    events=self.events,error=self.error,unfinished_frame=self.unfinished_frame,\n                    clock='client chunk-arrival wall; not token emission or GPU time')\n\n\ndef synthetic_fixture():\n    def chunk(delta,finish=None):\n        return dict(id='illustrative',object='chat.completion.chunk',model='topic04',\n                    choices=[dict(index=0,delta=delta,finish_reason=finish)])\n    payload=dict(model='topic04',messages=[dict(role='user',content='Say café ☕ briefly.')],\n                 max_tokens=16,temperature=0,stream=True,stream_options=dict(include_usage=True))\n    bodies=[chunk(dict(role='assistant',content='')),chunk(dict(content='Café ')),\n            chunk(dict(content='☕.')),chunk({},'stop'),\n            dict(choices=[],usage=dict(prompt_tokens=13,completion_tokens=4,total_tokens=17))]\n    # The synthetic usage counts are invented, not tokenizer measurements.\n    frames=[': heartbeat\\r\\n\\r\\n']+['data: '+json.dumps(b,ensure_ascii=False,separators=(',',':'))+'\\r\\n\\r\\n' for b in bodies]+['data: [DONE]\\r\\n\\r\\n']\n    arrival=[.01,.03,.10,.14,.15,.16,.17]\n    chunks=[]\n    for text,at_s in zip(frames,arrival):\n        raw=text.encode('utf-8')\n        # Always split CRLF; split UTF-8 continuation sequences when present.\n        cuts=sorted(set([len(raw)-3]+[i+1 for i,b in enumerate(raw) if b in (0xc3,0xe2)]))\n        previous=0\n        for end in cuts+[len(raw)]:\n            chunks.append(dict(at_s=at_s,hex=raw[previous:end].hex())); previous=end\n    stream=ChatStream()\n    for c in chunks: stream.feed(bytes.fromhex(c['hex']),c['at_s'])\n    complete=stream.finish()\n    prefix=ChatStream()\n    for c in chunks:\n        if c['at_s']>.10: break\n        prefix.feed(bytes.fromhex(c['hex']),c['at_s'])\n    partial=prefix.finish()\n    cancelled=ChatStream()\n    for c in chunks:\n        if c['at_s']>.10: break\n        cancelled.feed(bytes.fromhex(c['hex']),c['at_s'])\n    return dict(schema='topic04-sse-fixture-v1',kind='illustrative bytes, clocks and usage; no server or tokenizer executed',\n                request=payload,frames=frames,chunks=chunks,complete=complete,partial=partial,\n                cancelled=cancelled.finish(cancelled=True),source='https://html.spec.whatwg.org/multipage/server-sent-events.html')\n\n\"\"\"Canonical bounded paired smoke. Import is CPU-only; GPU work starts at run_lab().\"\"\"\nimport gc\nimport json\nimport math\nimport os\nimport statistics\nimport subprocess\nimport time\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nMODEL_ID='Qwen/Qwen2.5-0.5B-Instruct'\nREVISION='7ae557604adf67be50417f59c2c2f167def9a775'\nPROTOCOL='topic-03-paired-smoke-v3'\nTRIALS, MAX_NEW=3,24\nPROMPTS=[\n    dict(bucket='format',text='Reply with exactly one word: blue or orange. What color is the sky on a clear day?'),\n    dict(bucket='arithmetic',text='Compute 17 + 28. Give only the integer.'),\n    dict(bucket='instruction',text='Write exactly three comma-separated verbs for measuring a service.'),\n    dict(bucket='reasoning',text='A server has 2 GiB free and a new request needs 3 GiB. In one sentence, say whether it fits and why.'),\n]\n\n\ndef checkpoint(path,data):\n    path=Path(path); temporary=path.with_suffix(path.suffix+'.tmp')\n    temporary.write_text(json.dumps(data,indent=2,ensure_ascii=False,allow_nan=False)+'\\n',encoding='utf-8')\n    temporary.replace(path)\n\n\ndef resource_gate(require_idle=False):\n    mem=Path('/proc/meminfo')\n    if not mem.exists(): raise RuntimeError('Linux/WSL RAM gate unavailable; use supported CUDA runtime')\n    available=int(next(line.split()[1] for line in mem.read_text().splitlines() if line.startswith('MemAvailable:')))*1024\n    used=[int(x) for x in subprocess.check_output(['nvidia-smi','--query-gpu=memory.used','--format=csv,noheader,nounits'],text=True,timeout=10).split()]\n    if available < 24*2**30 or not used or max(used)>12288: raise RuntimeError('STOP: RAM/GPU resource gate')\n    if require_idle:\n        active=subprocess.check_output(['nvidia-smi','--query-compute-apps=pid','--format=csv,noheader,nounits'],text=True,timeout=10).strip()\n        if active: raise RuntimeError('STOP: GPU compute process already active')\n    return dict(available_ram_gib=available/2**30,device_used_mib=max(used))\n\n\ndef resolve_snapshot():\n    from huggingface_hub import snapshot_download\n    cache=Path(os.environ.get('LSM_MODEL_CACHE',Path.cwd()/'.cache/models')).expanduser()\n    try: snapshot=snapshot_download(MODEL_ID,revision=REVISION,cache_dir=str(cache),local_files_only=True)\n    except FileNotFoundError:\n        if os.environ.get('HF_HUB_OFFLINE')=='1': raise RuntimeError('Pinned offline snapshot missing; do not redownload')\n        snapshot=snapshot_download(MODEL_ID,revision=REVISION,cache_dir=str(cache))\n    return Path(snapshot)\n\n\n\n\"\"\"Bounded local API contract rehearsal. No GPU/import/network work at module import.\"\"\"\nimport atexit\nimport json\nimport os\nimport platform\nimport signal\nimport socket\nimport subprocess\nimport sys\nimport time\nfrom datetime import datetime,timezone\nfrom pathlib import Path\n\nPROTOCOL='topic-04-api-smoke-v3'\nSERVED_NAME='topic04'\nPROMPT='Reply with one short sentence explaining what an API server does.'\nRAW_PROMPT='Say hello briefly.'\n\n\ndef normalize_input_ids(encoded):\n    \"\"\"Transformers 4/5 may return IDs or a BatchEncoding; accept one sequence only.\"\"\"\n    value=encoded['input_ids'] if hasattr(encoded,'keys') else encoded\n    if isinstance(value,list) and len(value)==1 and isinstance(value[0],list): value=value[0]\n    if not isinstance(value,list) or not value or any(type(x) is not int or x<0 for x in value):\n        raise ValueError('Expected one nonempty integer input_ids sequence')\n    return value\n\n\ndef stop_owned_process_group(server,term_seconds=20,kill_seconds=10):\n    \"\"\"Stop only a POSIX session we created, including worker descendants.\"\"\"\n    try: os.killpg(server.pid,signal.SIGTERM)\n    except ProcessLookupError: pass\n    deadline=time.monotonic()+term_seconds\n    while time.monotonic()<deadline:\n        server.poll()\n        try: os.killpg(server.pid,0)\n        except ProcessLookupError: break\n        time.sleep(.1)\n    else:\n        try: os.killpg(server.pid,signal.SIGKILL)\n        except ProcessLookupError: pass\n    server.wait(timeout=kill_seconds)\n    deadline=time.monotonic()+kill_seconds\n    while time.monotonic()<deadline:\n        try: os.killpg(server.pid,0)\n        except ProcessLookupError: return\n        time.sleep(.1)\n    raise RuntimeError('Owned process group remains after teardown')\n\n\ndef run_lab(snapshot,output_path):\n    import httpx,openai,torch,vllm,transformers\n    from transformers import AutoTokenizer\n    if vllm.__version__!='0.29.0': raise RuntimeError('Pinned vLLM 0.29.0 required')\n    output_path=Path(output_path)\n    if output_path.exists(): raise FileExistsError('Preserve prior evidence: choose a fresh output path')\n    output_path.parent.mkdir(parents=True,exist_ok=True)\n    preflight=resource_gate(require_idle=True)\n    tokenizer=AutoTokenizer.from_pretrained(str(snapshot),local_files_only=True)\n    def prompt_record(text):\n        messages=[dict(role='user',content=text)]\n        ids=normalize_input_ids(tokenizer.apply_chat_template(messages,tokenize=True,add_generation_prompt=True))\n        return dict(messages=messages,rendered_chat=tokenizer.apply_chat_template(messages,tokenize=False,add_generation_prompt=True),\n                    input_token_ids=ids,prompt_tokens=len(ids),input_token_pieces=tokenizer.convert_ids_to_tokens(ids))\n    base='http://127.0.0.1:8000/v1'\n    with socket.socket() as probe:\n        if probe.connect_ex(('127.0.0.1',8000))==0: raise RuntimeError('Port 8000 occupied; do not stop an unowned server')\n    args=['serve',str(snapshot),'--served-model-name',SERVED_NAME,'--host','127.0.0.1','--port','8000',\n          '--dtype','float16','--max-model-len','2048','--max-num-seqs','4','--gpu-memory-utilization','0.45','--enforce-eager']\n    is_wsl=bool(os.environ.get('WSL_INTEROP')) or 'microsoft' in platform.release().lower()\n    env=os.environ.copy(); env['HF_HUB_OFFLINE']='1'; env['TRANSFORMERS_OFFLINE']='1'\n    if is_wsl: env['VLLM_USE_V2_MODEL_RUNNER']='0'; env['VLLM_USE_FLASHINFER_SAMPLER']='0'\n    data=dict(contract=PROTOCOL,status='running',model_id=MODEL_ID,revision=REVISION,served_name=SERVED_NAME,\n              utc_started=datetime.now(timezone.utc).isoformat(),vllm=vllm.__version__,openai_sdk=openai.__version__,\n              httpx=httpx.__version__,transformers=transformers.__version__,\n              torch=torch.__version__,cuda=torch.version.cuda,gpu=torch.cuda.get_device_name(0),\n              launch_args=['serve','<pinned-local-snapshot>']+args[2:],wsl_v1_runner_fallback=is_wsl,\n              resource_samples=[preflight],request_prompt=prompt_record(PROMPT),raw_prompt=prompt_record(RAW_PROMPT),\n              limits=dict(max_model_len=2048,max_num_seqs=4,gpu_memory_utilization=.45),\n              limitations=['single sequential API smoke, not throughput benchmark','first call may include lazy runtime initialization',\n                           'TCP/HTTP chunk boundaries are not generated token boundaries','client disconnect reclamation not measured'])\n    checkpoint(output_path,data)\n    log=output_path.with_name('server.local.log').open('x',encoding='utf-8')\n    server=None; stopped=False; client=None\n    def stop_server():\n        nonlocal stopped\n        if stopped: return\n        stopped=True\n        if server is not None:\n            stop_owned_process_group(server)\n        log.close()\n    atexit.register(stop_server)\n    try:\n        server=subprocess.Popen([sys.executable,'-m','vllm.entrypoints.cli.main']+args,\n                                env=env,stdout=log,stderr=subprocess.STDOUT,start_new_session=True)\n        begin=time.monotonic(); deadline=begin+300\n        while time.monotonic()<deadline:\n            if server.poll() is not None: raise RuntimeError('vLLM exited during startup; inspect private local log')\n            resource_gate()\n            try:\n                response=httpx.get(base+'/models',timeout=2,trust_env=False)\n                if response.status_code==200:\n                    advertised=[x['id'] for x in response.json()['data']]\n                    if SERVED_NAME not in advertised: raise RuntimeError('served name mismatch')\n                    data['advertised_models']=advertised; break\n            except httpx.RequestError: pass\n            time.sleep(2)\n        else: raise TimeoutError('Startup exceeded 300 s')\n        data['startup_s']=time.monotonic()-begin; checkpoint(output_path,data)\n        client=openai.OpenAI(base_url=base,api_key='EMPTY',timeout=120,max_retries=0,\n                            http_client=httpx.Client(trust_env=False))\n        request=dict(model=SERVED_NAME,messages=[dict(role='user',content=PROMPT)],max_tokens=48,temperature=0)\n        start=time.perf_counter(); answer=client.chat.completions.create(**request)\n        data['nonstream']=dict(request=request,wall_s=time.perf_counter()-start,finish_reason=answer.choices[0].finish_reason,\n                               usage=answer.usage.model_dump() if answer.usage else None,content=answer.choices[0].message.content)\n        if not data['nonstream']['content']: raise RuntimeError('Empty nonstream response')\n        checkpoint(output_path,data); resource_gate()\n        start=time.perf_counter(); first=None; pieces=[]; finish=None; chunks=0\n        stream=client.chat.completions.create(**request,stream=True)\n        try:\n            for chunk in stream:\n                chunks+=1\n                if chunks>512: raise RuntimeError('SDK chunk budget exceeded')\n                delta=chunk.choices[0].delta.content if chunk.choices else None\n                if delta:\n                    if first is None: first=time.perf_counter()-start\n                    pieces.append(delta)\n                if chunk.choices and chunk.choices[0].finish_reason: finish=chunk.choices[0].finish_reason\n        finally: stream.close()\n        data['sdk_stream']=dict(first_content_s=first,wall_s=time.perf_counter()-start,finish_reason=finish,\n                                content=''.join(pieces),sdk_chunks=chunks)\n        if first is None or finish is None: raise RuntimeError('Incomplete SDK stream')\n        checkpoint(output_path,data); resource_gate()\n        payload=dict(model=SERVED_NAME,messages=[dict(role='user',content=RAW_PROMPT)],max_tokens=16,temperature=0,\n                     stream=True,stream_options=dict(include_usage=True))\n        parser=ChatStream(); raw_chunks=[]; start=time.perf_counter()\n        with httpx.stream('POST',base+'/chat/completions',json=payload,timeout=120,trust_env=False) as response:\n            response.raise_for_status()\n            media=response.headers.get('content-type','').split(';')[0].lower()\n            if media!='text/event-stream': raise RuntimeError('Wrong SSE content type')\n            for raw in response.iter_bytes():\n                at=time.perf_counter()-start\n                raw_chunks.append(dict(at_s=at,hex=raw.hex()))\n                parser.feed(raw,at)\n        record=parser.finish(); record['request']=payload; record['http_status']=response.status_code\n        record['content_type']=media; record['chunks']=raw_chunks\n        data['raw_sse']=record; checkpoint(output_path,data)\n        if record['status']!='complete' or not record['content']: raise RuntimeError('Incomplete raw SSE')\n        negative={}\n        try: client.chat.completions.create(model='not-the-served-model',messages=[dict(role='user',content='hi')],max_tokens=1)\n        except openai.APIStatusError as exc: negative['wrong_model']=dict(status=exc.status_code)\n        long_text=' blue'*2300\n        rendered=normalize_input_ids(tokenizer.apply_chat_template([dict(role='user',content=long_text)],tokenize=True,add_generation_prompt=True))\n        if not 2048<len(rendered)<8192: raise RuntimeError('Bounded over-context fixture invalid')\n        try: client.chat.completions.create(model=SERVED_NAME,messages=[dict(role='user',content=long_text)],max_tokens=16)\n        except openai.APIStatusError as exc: negative['over_context']=dict(status=exc.status_code,rendered_prompt_tokens=len(rendered))\n        if set(negative)!={'wrong_model','over_context'} or any(not 400<=r['status']<500 for r in negative.values()):\n            raise RuntimeError('Expected 4xx contract not met')\n        data['negative']=negative; data['status']='complete'; checkpoint(output_path,data)\n    except BaseException as exc:\n        data['status']='failed'; data['error_type']=type(exc).__name__; raise\n    finally:\n        try:\n            if client is not None: client.close()\n            stop_server(); atexit.unregister(stop_server)\n            data['teardown']=dict(owned_process_group_stopped=True,server_returncode=None if server is None else server.returncode)\n            data['resource_samples'].append(resource_gate(require_idle=True))\n        except BaseException as exc:\n            data['status']='failed'; data['cleanup_error_type']=type(exc).__name__\n            raise\n        finally:\n            data['utc_finished']=datetime.now(timezone.utc).isoformat(); checkpoint(output_path,data)\n    return data\n\n\ndef validate_artifact(data):\n    import math\n    def require(value,message):\n        if not value: raise ValueError(message)\n    require(data.get('contract')==PROTOCOL and data.get('status')=='complete','incomplete/wrong protocol')\n    require(data.get('revision')==REVISION and data.get('model_id')==MODEL_ID,'model identity')\n    require(data.get('vllm')=='0.29.0' and all(isinstance(data.get(key),str) and data[key] for key in ('openai_sdk','httpx','transformers','torch','cuda','gpu')),'runtime pins/identity')\n    require(data.get('served_name')==SERVED_NAME and SERVED_NAME in data.get('advertised_models',[]),'served identity')\n    for key,text in [('request_prompt',PROMPT),('raw_prompt',RAW_PROMPT)]:\n        prompt=data[key]\n        require(prompt['messages']==[dict(role='user',content=text)],'literal request')\n        require(prompt['prompt_tokens']==len(prompt['input_token_ids'])==len(prompt['input_token_pieces'])>0,'prompt count')\n        require(all(type(x) is int and x>=0 for x in prompt['input_token_ids']),'ID types')\n    for key in ('nonstream','sdk_stream'):\n        result=data[key]\n        require(isinstance(result.get('content'),str) and bool(result['content']),'content required')\n        require(result.get('finish_reason') in ('stop','length'),'finish reason')\n        require(math.isfinite(result['wall_s']) and result['wall_s']>0,'wall clock')\n    require(0<data['sdk_stream']['first_content_s']<=data['sdk_stream']['wall_s'],'first content clock')\n    raw=data['raw_sse']; replay=ChatStream()\n    for chunk in raw['chunks']: replay.feed(bytes.fromhex(chunk['hex']),chunk['at_s'])\n    reconstructed=replay.finish()\n    require(all(raw[key]==value for key,value in reconstructed.items()),'raw byte replay mismatch')\n    require(raw['http_status']==200 and raw['content_type']=='text/event-stream','SSE HTTP boundary')\n    require(raw['status']=='complete' and raw['done'] and raw['content'],'partial stream is not success')\n    require(raw['request']['messages']==data['raw_prompt']['messages'],'raw prompt identity')\n    require(raw['usage'] is not None and raw['usage']['prompt_tokens']==data['raw_prompt']['prompt_tokens'],'raw usage vs tokenizer')\n    usage=data['nonstream']['usage']\n    require(usage['prompt_tokens']==data['request_prompt']['prompt_tokens'] and usage['total_tokens']==usage['prompt_tokens']+usage['completion_tokens'],'nonstream usage')\n    require(data['teardown']['owned_process_group_stopped'],'owned server teardown')\n    require(set(data['negative'])=={'wrong_model','over_context'},'negative probes')\n    require(all(400<=row['status']<500 for row in data['negative'].values()),'negative 4xx')\n    require(2048<data['negative']['over_context']['rendered_prompt_tokens']<8192,'bounded token rejection')\n    return True\n",lab_namespace)
print('Read-only headroom:',lab_namespace['resource_gate'](require_idle=True))
MODEL_DIR=lab_namespace['resolve_snapshot']()
print('Pinned local snapshot resolved; do not publish its filesystem path.')


## 3. Startup → identity → one response → stream → negative probes → teardown

The complete editable helper above binds only 127.0.0.1:8000 and refuses an occupied port; it never kills an unowned service. Startup has a 300 s deadline, requests 120 s, raw stream 256 KiB/4096 events. Max retries is zero so failures are not hidden by automatic resends. Evidence checkpoints follow each stage.

Read /v1/models, then compare the advertised name with the exact model field. The nonstream/SDK prompt cap is 48, raw probe cap 16. The negative context request is tokenizer-counted and bounded below 8192 input IDs; a real 4xx is expected. Timeout, server failure or unexpected success is not an acceptable negative-test result.

Finally closes the client and terminates **only the process group this helper owns**, then verifies no compute process remains. Stop immediately on OOM, resource pressure or failed teardown; preserve partial JSON and the private server.local.log, diagnose the failing stage, and do not blindly restart all labs.


In [ ]:
from pathlib import Path
from datetime import datetime,timezone
ATTEMPT=Path('evidence')/datetime.now(timezone.utc).strftime('topic04-%Y%m%dT%H%M%S%fZ')
ATTEMPT.mkdir(parents=True,exist_ok=False)
RUN=lab_namespace['run_lab'](MODEL_DIR,ATTEMPT/'service.json')
assert lab_namespace['validate_artifact'](RUN)
print('Validated API smoke:',ATTEMPT/'service.json')
print('Owned server stopped:',RUN['teardown'])


## 4. Deliver a boundary-aware analysis

Copy validated JSON to evidence/topic-04-service.json; keep the fresh original attempt. Submit evidence/topic-04-analysis.md with: (1) literal request → SDK/HTTP/SSE → template/tokenizer → engine/scheduler → worker/GPU → client boundary map; (2) actual prompt+output budget and why 45% memory is not a throughput promise; (3) first-event vs first-content vs finish vs terminal stream distinctions; (4) one observed negative-test failure path and mitigation; (5) a concrete next test for buffering, cancellation reclamation or concurrent load.

The CPU validator replays every recorded byte and checks usage, IDs, identity, completion, expected 4xx and teardown. It does not establish production throughput, proxy correctness, engine cancellation or judge readiness. Raw server logs stay local and require sanitization before private sharing. Standalone self-check: python scripts/validate_topic04.py evidence/topic-04-service.json in the public template.
